# Genetic-oscillator inference: PST training

Ten PST training runs of 3,000 epochs for genetic-oscillator parameter inference (Figure 3). The results are used by [Oscillator-analysis.ipynb](Oscillator-analysis.ipynb).

Training writes `experiment_results_pst_3000_stateless_categorical.pkl` and completed per-run checkpoints in `pst_3000_stateless_categorical_checkpoints/`. With `OVERWRITE_FINAL = False`, a complete aggregate result is reused. If the aggregate is absent, completed checkpoints are reused and the remaining runs are trained. See the [run guide](../docs/RUNNING.md#oscillator-training-and-analysis) for a fresh training run.

Run from the `Oscillator/` directory using the `PST (Python 3.11)` kernel. See [Running the notebooks](../docs/RUNNING.md) for execution settings and result reuse.

**Authors:** Jose M. G. Vilar$^{1,2}$ and Leonor Saiz$^{3}$  
$^1$ Biofisika Institute (CSIC, UPV/EHU), University of the Basque Country, P.O. Box 644, 48080 Bilbao, Spain  
$^2$ IKERBASQUE, Basque Foundation for Science, 48011 Bilbao, Spain  
$^3$ Department of Biomedical Engineering, University of California, 451 East Health Sciences Drive, Davis, CA 95616, USA

**Preprint:** J. M. G. Vilar and L. Saiz, *Propensity Straight-Through Gradients for Discrete Stochastic Systems*, arXiv:2608.25631 [q-bio.QM] (2026).  
**DOI:** [https://doi.org/10.48550/arXiv.2608.25631](https://doi.org/10.48550/arXiv.2608.25631)


In [ ]:
import gc

# Silence C/C++ (stderr/stdout) just for the TF import & warm-up.
import os, ctypes, contextlib

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # Set GPU device


libc = ctypes.CDLL(None)

@contextlib.contextmanager
def silence_native_output():
    # Save original fds
    saved_out = os.dup(1)
    saved_err = os.dup(2)
    try:
        devnull = os.open(os.devnull, os.O_WRONLY)
        libc.fflush(None)                    # flush C stdio buffers
        os.dup2(devnull, 1)                  # redirect stdout -> /dev/null
        os.dup2(devnull, 2)                  # redirect stderr -> /dev/null
        os.close(devnull)
        yield
    finally:
        libc.fflush(None)
        os.dup2(saved_out, 1)                # restore stdout
        os.dup2(saved_err, 2)                # restore stderr
        os.close(saved_out); os.close(saved_err)

# Set TF verbosity before import (works for Python/absl logs after init)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

with silence_native_output():
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        try: 
            tf.config.experimental.set_memory_growth(g, True)
            print(f"Memory growth enabled for {g} GPU(s)")
        except RuntimeError as e:
            print(f"Memory growth setting failed: {e}")
    # Optional: XLA/GPU warm-up so later compilations don't log either
    @tf.function(jit_compile=True)
    def _warmup(x): return x + 1
    _ = _warmup(tf.ones([1]))

# Clamp Python-level loggers too (just in case)
tf.get_logger().setLevel("ERROR")
import absl.logging as absl_logging
absl_logging.set_verbosity(absl_logging.ERROR)
print("TensorFlow imported quietly")


import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import time
import os
import gc
import pickle

from pathlib import Path

# ============================================================
# PST STATELESS-CATEGORICAL 3000-EPOCH RUN CONFIGURATION
# ============================================================
# Forward sampler: tf.random.stateless_categorical
# Backward surrogate: tf.nn.softmax(logits)
# No explicit Gumbel variable or temperature is used.
OUTPUT_FILE = "experiment_results_pst_3000_stateless_categorical.pkl"
CHECKPOINT_DIR = "pst_3000_stateless_categorical_checkpoints"

N_RUNS = 10
NUM_EPOCHS = 3000
STATS_PCT = 25.0
BASE_SEED = 43

# Reuse a complete final result unless OVERWRITE_FINAL is enabled.
OVERWRITE_FINAL = False

SCRIPT_DIR = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd()
)


def output_path(filename):
    return SCRIPT_DIR / filename


def atomic_pickle_dump(value, filepath):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    temporary = filepath.with_suffix(filepath.suffix + ".tmp")
    with open(temporary, "wb") as handle:
        pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary, filepath)


def validate_final_results(results):
    required = {
        "histories",
        "run_stats",
        "init_multipliers",
        "final_params",
        "global_stats",
        "aggregated_stats",
        "N",
        "stats_pct",
    }
    missing = required.difference(results)
    if missing:
        raise ValueError(f"Final PST file is missing keys: {sorted(missing)}")
    if int(results["N"]) != N_RUNS:
        raise ValueError(
            f"Final PST file contains {results['N']} runs; expected {N_RUNS}."
        )
    if results.get("forward_sampler") != "tf.random.stateless_categorical":
        raise ValueError(
            "Final PST file was not generated with the stateless categorical sampler."
        )
    lengths = [len(history["epoch"]) for history in results["histories"]]
    if lengths != [NUM_EPOCHS] * N_RUNS:
        raise ValueError(
            f"Final PST histories have lengths {lengths}; "
            f"expected {NUM_EPOCHS} epochs for every run."
        )


# ========================================================================
# GLOBAL DEFINITIONS
# ========================================================================

# ----------------------------------------
# 1. Define the Vilar oscillator parameters
# ----------------------------------------
k_true = np.array([
    1.0,    # params["gamma_A"]
    50.0,   # params["theta_A"]
    1.0,    # params["gamma_R"]
    100.0,  # params["theta_R"]
    50.0,   # params["alpha_A"]
    500.0,  # params["alpha_A'"]
    0.01,   # params["alpha_R"]
    50.0,   # params["alpha_R'"]
    50.0,   # params["beta_A"]   -> TRAIN (index 8)
    5.0,    # params["beta_R"]   -> TRAIN (index 9)
    10.0,   # params["delta_MA"]
    0.5,    # params["delta_MR"]
    1.0,    # params["delta_A"]  -> TRAIN (index 12)
    0.2,    # params["delta_R"]  -> TRAIN (index 13)
    2.0/1,  # params["gamma_C"]  -> TRAIN (index 14)
    1.0     # params["delta_A"]  -> TRAIN (index 15)
], dtype=np.float32)

# ---------------------------------------------------
# 2. Build the stoichiometry matrix S (16 × 9, float32)
# ---------------------------------------------------
S = tf.constant([
    #    DA    DAp   DR    DRp   MA    MR     A     R     C
    [-1.0, +1.0,  0.0,  0.0,  0.0,  0.0, -1.0,  0.0,  0.0], # 0
    [ 1.0, -1.0,  0.0,  0.0,  0.0,  0.0,  1.0,  0.0,  0.0], # 1
    [ 0.0,  0.0, -1.0, +1.0,  0.0,  0.0, -1.0,  0.0,  0.0], # 2
    [ 0.0,  0.0, +1.0, -1.0,  0.0,  0.0,  1.0,  0.0,  0.0], # 3
    [ 0.0,  0.0,  0.0,  0.0, +1.0,  0.0,  0.0,  0.0,  0.0], # 4
    [ 0.0,  0.0,  0.0,  0.0, +1.0,  0.0,  0.0,  0.0,  0.0], # 5
    [ 0.0,  0.0,  0.0,  0.0,  0.0, +1.0,  0.0,  0.0,  0.0], # 6
    [ 0.0,  0.0,  0.0,  0.0,  0.0, +1.0,  0.0,  0.0,  0.0], # 7
    [ 0.0,  0.0,  0.0,  0.0,  0.0,  0.0, +1.0,  0.0,  0.0], # 8
    [ 0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0, +1.0,  0.0], # 9
    [ 0.0,  0.0,  0.0,  0.0, -1.0,  0.0,  0.0,  0.0,  0.0], # 10
    [ 0.0,  0.0,  0.0,  0.0,  0.0, -1.0,  0.0,  0.0,  0.0], # 11
    [ 0.0,  0.0,  0.0,  0.0,  0.0,  0.0, -1.0,  0.0,  0.0], # 12
    [ 0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0, -1.0,  0.0], # 13
    [ 0.0,  0.0,  0.0,  0.0,  0.0,  0.0, -1.0, -1.0, +1.0], # 14
    [ 0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0, +1.0, -1.0], # 15
], dtype=tf.float32)

# -------------------------------------------------------
# 3. Propensity function
# -------------------------------------------------------
@tf.function
def propensity_fn(states, k):
    X = tf.maximum(states, 1e-10)
    DA  = X[:, 0]; DAp = X[:, 1]; DR  = X[:, 2]; DRp = X[:, 3]
    MA  = X[:, 4]; MR  = X[:, 5]; A   = X[:, 6]; R   = X[:, 7]; C = X[:, 8]
    return tf.stack([
        k[0]  * DA  * A, k[1]  * DAp, k[2]  * DR  * A, k[3]  * DRp,
        k[4]  * DA,      k[5]  * DAp, k[6]  * DR,      k[7]  * DRp,
        k[8]  * MA,      k[9]  * MR,  k[10] * MA,      k[11] * MR,
        k[12] * A,       k[13] * R,   k[14] * A   * R, k[15] * C
    ], axis=1)

# -----------------------------------------------------------------------------------
# 4. Simulation functions
# -----------------------------------------------------------------------------------
def pst_estimator_stateless(logits, seed):
    """
    Stateless propensity-based straight-through (PST) estimator.

    Forward pass:
        exact categorical sampling directly from the supplied logits.

    Backward pass:
        normalized reaction propensities, softmax(logits).
    """
    logits_shape = tf.shape(logits)
    num_reactions = logits_shape[-1]

    # tf.random.stateless_categorical expects 2D inputs
    # [batch_size, num_classes].
    flat_logits = tf.reshape(logits, [-1, num_reactions])

    hard_idx_flat = tf.random.stateless_categorical(
        flat_logits,
        num_samples=1,
        seed=seed,
    )

    hard_idx = tf.reshape(hard_idx_flat, logits_shape[:-1])

    y_hard = tf.one_hot(
        hard_idx,
        depth=num_reactions,
        dtype=logits.dtype,
    )

    y_backward = tf.nn.softmax(logits, axis=-1)

    return y_backward + tf.stop_gradient(y_hard - y_backward)

def simulate_parallel_pst_stateless(batch_initial_states, num_steps, k, S_matrix, seed):
    num_simulations = tf.shape(batch_initial_states)[0]
    time_t = tf.zeros([num_simulations, 1], tf.float32)
    states_ta = tf.TensorArray(tf.float32, size=num_steps + 1)
    times_ta  = tf.TensorArray(tf.float32, size=num_steps + 1)
    states_ta = states_ta.write(0, batch_initial_states)
    times_ta  = times_ta.write(0, time_t)

    t = tf.constant(0)
    states = batch_initial_states

    def cond(t, states, time_t, states_ta, times_ta, seed):
        return t < num_steps

    def body(t, states, time_t, states_ta, times_ta, seed):
        seeds = tf.random.experimental.stateless_split(seed, 3)
        seed      = seeds[0]
        seed_time = seeds[1]
        seed_reaction = seeds[2]
    
        prop = propensity_fn(states, k)
        a0 = tf.reduce_sum(prop, axis=1, keepdims=True)
        active = a0 > 0.0
        safe_a0 = tf.where(active, a0, tf.ones_like(a0))

        u_time = tf.random.stateless_uniform(
            [num_simulations, 1],
            seed=seed_time,
            minval=1e-20,
            maxval=1.0,
            dtype=prop.dtype,
        )
        dt = tf.where(
            active,
            -tf.math.log(u_time) / safe_a0,
            tf.zeros_like(u_time),
        )

        # Preserve exactly zero propensities. Deadlocked rows use a temporary
        # deterministic category and are masked before the state update.
        fallback = tf.one_hot(
            tf.zeros([num_simulations], dtype=tf.int32),
            depth=tf.shape(prop)[-1],
            dtype=prop.dtype,
        )
        sampling_prop = tf.where(active, prop, fallback)
        positive = sampling_prop > 0.0
        finite_log = tf.math.log(
            tf.where(positive, sampling_prop, tf.ones_like(sampling_prop))
        )
        logits = tf.where(
            positive,
            finite_log,
            tf.constant(float("-inf"), dtype=prop.dtype),
        )

        weights = pst_estimator_stateless(logits, seed_reaction)
        weights = tf.where(active, weights, tf.zeros_like(weights))
    
        delta = tf.matmul(weights, S_matrix)
        new_states = tf.maximum(states + delta, 0.0)
        new_time = time_t + dt
    
        states_ta = states_ta.write(t + 1, new_states)
        times_ta  = times_ta.write(t + 1, new_time)
        return t + 1, new_states, new_time, states_ta, times_ta, seed


    _, _, _, states_ta, times_ta, _ = tf.while_loop(
        cond, body,
        loop_vars=[t, states, time_t, states_ta, times_ta, seed],
        shape_invariants=[t.shape, tf.TensorShape([None, 9]), tf.TensorShape([None, 1]),
                          tf.TensorShape([]), tf.TensorShape([]), tf.TensorShape([2])]
    )

    states = tf.transpose(states_ta.stack(), [1, 0, 2])
    times  = tf.transpose(times_ta.stack(),  [1, 0, 2])
    return states, times

# 2. Create the JIT-compiled version (Fast, No Gradients)
simulate_parallel_fast = tf.function(jit_compile=True)(simulate_parallel_pst_stateless)

# 3. Create the standard version (Supports Gradients)
simulate_parallel_grads = tf.function(jit_compile=False)(simulate_parallel_pst_stateless)

# -----------------------------------------------------------------------------------
# 5. Helper function to assemble k parameter vector
# -----------------------------------------------------------------------------------
def assemble_k_from_params(beta_A, beta_R, delta_A, delta_R, gamma_C):
    """Assemble full k parameter vector from individual parameters"""
    k_const = tf.constant(k_true, dtype=tf.float32)
    return tf.concat([
        k_const[:8], [beta_A], [beta_R], 
        k_const[10:12], [delta_A], [delta_R], 
        [gamma_C], [delta_A]
    ], axis=0)



# -----------------------------------------------------------------------------------
# 7. Statistics computation helper
# -----------------------------------------------------------------------------------
def compute_run_statistics(history, stats_pct=25.0):
    """
    Compute statistics from a single run's history.

    Parameters:
    -----------
    history : dict
        Training history dictionary
    stats_pct : float
        Percentage of final epochs to use for statistics (default: 25%)

    Returns:
    --------
    dict : Statistics including median params, median mape, medmape
    """
    param_keys = ['beta_A', 'beta_R', 'delta_A', 'delta_R', 'gamma_C']
    true_vals = np.array([50.0, 5.0, 1.0, 0.2, 2.0])

    # Select last X% of epochs
    num_epochs = len(history['loss'])
    num_to_select = max(1, int(num_epochs * stats_pct / 100.0))
    idx = np.arange(num_epochs)[-num_to_select:]

    # Compute geometric median for each parameter
    median_params = {}
    median_params_list = []
    for var in param_keys:
        sample = np.array(history[var])[idx]
        log_sample = np.log(sample + 1e-20)
        geo_median = np.exp(np.mean(log_sample))
        median_params[var] = geo_median
        median_params_list.append(geo_median)

    # Compute median MAPE from the history
    mape_sample = np.array(history['mape'])[idx]
    median_mape = np.exp(np.mean(np.log(mape_sample + 1e-20)))

    # Compute medmape (MAPE of median params)
    esvals = np.array(median_params_list)
    medmape = 100 * np.mean(np.abs(true_vals - esvals) / true_vals)

    return {
        'median_params': median_params,
        'median_mape': median_mape,
        'medmape': medmape,
        'num_selected': num_to_select
    }


# ========================================================================
# MAIN INFERENCE FUNCTION
# ========================================================================

def InferenceOscillator(init_multipliers=None, seed=43, verbose=False, 
                        stats_pct=25.0, **kwargs):
    """
    Vilar Oscillator parameter inference using gradient-based optimization.

    Parameters:
    -----------
    init_multipliers : array-like of length 5, optional
        Multipliers for initial parameter values [beta_A, beta_R, delta_A, delta_R, gamma_C].
        Each true parameter is multiplied by the corresponding multiplier.
        Default: [0.25, 4.0, 4.0, 0.25, 0.25]
    seed : int
        Random seed for reproducibility (default: 43)
    verbose : bool
        If True, prints detailed ratios/params and processing messages (default: False)
    stats_pct : float
        Percentage of final epochs to use for statistics (default: 25%)
    **kwargs : dict
        Optional parameters to override defaults

    Returns:
    --------
    history : dict
        Training history
    final_params : dict
        Final learned parameters
    run_stats : dict
        Per-run statistics
    """

    # Set default initial multipliers
    if init_multipliers is None:
        init_multipliers = np.array([0.25, 4.0, 4.0, 0.25, 0.25], dtype=np.float32)
    else:
        init_multipliers = np.array(init_multipliers, dtype=np.float32)

    assert len(init_multipliers) == 5, "init_multipliers must have exactly 5 elements"

    print(f"\n=== Running Inference Oscillator (seed={seed}) ===")
    print(f"Initial multipliers: {init_multipliers}")

    # Set random seeds
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.random.experimental.get_global_generator().reset_from_seed(seed)

    # -----------------------------------------------------------------------------------
    # Target map generation
    # -----------------------------------------------------------------------------------
    def generate_target_map(initial_state_np, num_start_points, len_reference_traj, len_segment, 
                          M_train, verbose=False, ref_traj_filename='reference_trajectory.npz'):

        # Load or generate reference trajectory
        if False:#os.path.exists(ref_traj_filename):
            print(f"Loading existing reference trajectory from {Path(ref_traj_filename).name}...")
            with np.load(ref_traj_filename) as data:
                ref_sim_states = tf.constant(data['states'], dtype=tf.float32)
                ref_sim_times = tf.constant(data['times'], dtype=tf.float32)
        else:
            print(f"Generating a long reference trajectory of {len_reference_traj} steps...")
            seed2 = tf.constant([seed, 11], dtype=tf.int32)
            ref_sim_states, ref_sim_times = simulate_parallel_fast(
                batch_initial_states=initial_state_np[None, :],
                num_steps=len_reference_traj, k=tf.constant(k_true), S_matrix=S, seed=seed2
            )
            print(f"Saving reference trajectory to {Path(ref_traj_filename).name}...")
            np.savez(ref_traj_filename, states=ref_sim_states.numpy(), times=ref_sim_times.numpy())

        reference_states = tf.squeeze(ref_sim_states, axis=0)
        reference_times = tf.squeeze(ref_sim_times, axis=0)

        print(f"Generating target map from {num_start_points} starting points...")
        burn_in = 300_000
        max_start_excl = len_reference_traj - len_segment + 1  # exclusive
        N = tf.cast(num_start_points, tf.int32)        
        # Strata edges over valid start-index range [burn_in, max_start_excl)
        edges = tf.linspace(tf.cast(burn_in, tf.float32), tf.cast(max_start_excl, tf.float32), N + 1)  # [N+1]
        left, right = edges[:-1], edges[1:]  # [N], [N]
        u = tf.random.uniform([N], 0.0, 1.0)  # jitter per stratum
        start_indices = tf.cast(tf.floor(left + u * (right - left)), tf.int32)
        start_indices = tf.clip_by_value(start_indices, burn_in, max_start_excl - 1)


        initial_states_for_map = tf.gather(reference_states, start_indices)

        print("  Generating target trajectories (vectorized)...")
        
        # Indices for each segment: [num_start_points, len_segment+1]
        offsets = tf.range(len_segment + 1, dtype=tf.int32)[None, :]            # [1, L]
        idx = start_indices[:, None] + offsets                                   # [N, L]
        
        # Gather all segments in one shot
        final_target_states = tf.gather(reference_states, idx)                   # [N, L, 9]
        final_target_times_raw = tf.gather(reference_times, idx)                 # [N, L, 1]
        
        # Re-zero each segment's time so t0 = 0.
        final_target_times = final_target_times_raw - final_target_times_raw[:, :1, :]


        # Create initial states for training by repeating M_train times
        initial_states_for_training = tf.repeat(initial_states_for_map[:, None, :], repeats=M_train, axis=1)

        return initial_states_for_training, final_target_states, final_target_times, reference_states, reference_times

    # -----------------------------------------------------------------------------------
    # Loss function
    # -----------------------------------------------------------------------------------
    def compute_loss(sim_states, target_states, sim_times, target_times, lambda_t):
        """Computes loss using rate of change comparison"""
        epsilon = 1e-9

        def _get_rate(states, times, feature_idx):
            """Calculates a stable rate of change from the initial state."""
            mean_final_state = tf.reduce_mean(states[:, -100:, feature_idx], axis=1, keepdims=True)
            mean_final_time = tf.reduce_mean(times[:, -100:, 0], axis=1, keepdims=True)            
            delta_state = mean_final_state - states[:, 0:1, feature_idx]
            delta_time = mean_final_time - times[:, 0:1, 0]            
            return delta_state / (delta_time + epsilon)

        loss_A = tf.reduce_mean(tf.square(_get_rate(sim_states, sim_times, 6) - _get_rate(target_states, target_times, 6)))
        loss_R = tf.reduce_mean(tf.square(_get_rate(sim_states, sim_times, 7) - _get_rate(target_states, target_times, 7)))
        loss_C = tf.reduce_mean(tf.square(_get_rate(sim_states, sim_times, 8) - _get_rate(target_states, target_times, 8)))

        loss_t = tf.reduce_mean(tf.square(
            (sim_times[:, -1, 0] - sim_times[:, 0, 0]) -
            (target_times[:, -1, 0] - target_times[:, 0, 0])
        ))

        return loss_A + loss_R + loss_C + lambda_t * loss_t

    # -----------------------------------------------------------------------------------
    # Main execution parameters
    # -----------------------------------------------------------------------------------
    kk = kwargs.get('kk', 8)

    default_params = {
        'NUM_START_POINTS': 64*64*4*4*4,
        'LEN_REFERENCE_TRAJ': 2*4*300000,
        'T_segment': 150,
        'M_TRAIN': (50*2)//4,
        'num_epochs': 3000*kk//1,
        'batch_size': (64*64//4)*8*1,
        'initial_learning_rate': 1e-1,
        'learning_rate_decay_factor': 0.9,
        'lambda_t': 0,
        'kk': kk
    }

    
    params = {**default_params, **kwargs}

    NUM_START_POINTS = params['NUM_START_POINTS']
    LEN_REFERENCE_TRAJ = params['LEN_REFERENCE_TRAJ']
    T_segment = params['T_segment']
    M_TRAIN = params['M_TRAIN']
    num_epochs = params['num_epochs']
    batch_size = params['batch_size']
    initial_learning_rate = params['initial_learning_rate']
    learning_rate_decay_factor = params['learning_rate_decay_factor']
    lambda_t = params['lambda_t']
    kk = params['kk']

    print(f"Parameters: {NUM_START_POINTS} start points, {M_TRAIN} train sims, {num_epochs} epochs")

    # -----------------------------------------------------------------------------------
    # Generate target map and setup training
    # -----------------------------------------------------------------------------------
    initial_state = np.array([1, 0, 1, 0, 0, 0, 0, 0, 0], dtype=np.float32)
    map_initial_states, map_target_trajectories, map_target_times, ref_traj_states, ref_traj_times = generate_target_map(
        initial_state, NUM_START_POINTS, LEN_REFERENCE_TRAJ, T_segment, M_TRAIN, verbose
    )
    print("Target map generated.")

    # Setup trainable variables in LOG-SPACE with init_multipliers
    k_const = tf.constant(k_true, dtype=tf.float32)
    beta_A_init = np.float32(init_multipliers[0] * k_true[8])
    beta_R_init = np.float32(init_multipliers[1] * k_true[9])
    delta_A_init = np.float32(init_multipliers[2] * k_true[12])
    delta_R_init = np.float32(init_multipliers[3] * k_true[13])
    gamma_C_init = np.float32(init_multipliers[4] * k_true[14])

    print(f"Initial params: beta_A={beta_A_init:.4f}, beta_R={beta_R_init:.4f}, "
          f"delta_A={delta_A_init:.4f}, delta_R={delta_R_init:.4f}, gamma_C={gamma_C_init:.4f}")

    log_beta_A = tf.Variable(tf.math.log(beta_A_init), dtype=tf.float32, name="log_beta_A")
    log_beta_R = tf.Variable(tf.math.log(beta_R_init), dtype=tf.float32, name="log_beta_R")
    log_delta_A = tf.Variable(tf.math.log(delta_A_init), dtype=tf.float32, name="log_delta_A")
    log_delta_R = tf.Variable(tf.math.log(delta_R_init), dtype=tf.float32, name="log_delta_R")
    log_gamma_C = tf.Variable(tf.math.log(gamma_C_init), dtype=tf.float32, name="log_gamma_C")
    trainable_vars = [log_beta_A, log_beta_R, log_delta_A, log_delta_R, log_gamma_C]

    @tf.function
    def assemble_k(log_beta_A_var, log_beta_R_var, log_delta_A_var, log_delta_R_var, log_gamma_C_var):
        return tf.concat([
            k_const[:8], [tf.exp(log_beta_A_var)], [tf.exp(log_beta_R_var)], 
            k_const[10:12], [tf.exp(log_delta_A_var)], [tf.exp(log_delta_R_var)], 
            [tf.exp(log_gamma_C_var)], [tf.exp(log_delta_A_var)]
        ], axis=0)

    # -----------------------------------------------------------------------------------
    # Training loop
    # -----------------------------------------------------------------------------------
    optimizer = tf.keras.optimizers.RMSprop(learning_rate=initial_learning_rate)

    history = {
        'epoch': [], 'loss': [], 'rpd': [], 'mape': [], 'beta_A': [], 'beta_R': [], 
        'delta_A': [], 'delta_R': [], 'gamma_C': [], 'ratio_bA_dA': [], 'ratio_bR_dR': [], 'ratio_dA_dR': [],
        'medmape': []
    }

    bA_true, bR_true, dA_true, dR_true, gC_true = k_true[8], k_true[9], k_true[12], k_true[13], k_true[14]
    true_ratios = {'bA_dA': bA_true / dA_true, 'bR_dR': bR_true / dR_true, 'dA_dR': dA_true / dR_true}
    true_log_params = tf.math.log(tf.constant([bA_true, bR_true, dA_true, dR_true, gC_true]))

    def calculate_rpd(log_bA, log_bR, log_dA, log_dR, log_gC):
        learned_log_params = tf.stack([log_bA, log_bR, log_dA, log_dR, log_gC])
        return tf.norm(learned_log_params - true_log_params)

    def calculate_mape(learned_params, true_params):
        epsilon = 1e-9
        ape = tf.abs((learned_params - true_params) / (true_params + epsilon)) * 100
        return tf.reduce_mean(ape)

    print(f"\nStarting PST training (M_TRAIN = {M_TRAIN}, 3000-epoch protocol)...")
    start_time = time.time()

    true_params = tf.constant([bA_true, bR_true, dA_true, dR_true, gC_true], dtype=tf.float32)
    true_log_params = tf.math.log(true_params)
    true_log_params_list = tf.unstack(true_log_params)  # graph-safe iteration helper
    log10 = tf.math.log(tf.constant(10.0, dtype=tf.float32))
    
    @tf.function(reduce_retracing=True)
    def train_step(batch_initial_states, batch_target_states, batch_target_times):
        # Optional shape constraints
        batch_initial_states = tf.ensure_shape(batch_initial_states, [batch_size, M_TRAIN, 9])
        batch_target_states  = tf.ensure_shape(batch_target_states,  [batch_size, T_segment + 1, 9])
        batch_target_times   = tf.ensure_shape(batch_target_times,   [batch_size, T_segment + 1, 1])
    
        with tf.GradientTape() as tape:
            k_learn = assemble_k(log_beta_A, log_beta_R, log_delta_A, log_delta_R, log_gamma_C)
    
            super_batch_initial_states = tf.repeat(batch_initial_states[:, 0, :], repeats=M_TRAIN, axis=0)
            
            seed2 = tf.constant([seed, epoch], dtype=tf.int32)    
            super_sim_states, super_sim_times = simulate_parallel_grads(
                batch_initial_states=super_batch_initial_states,
                num_steps=T_segment,
                k=k_learn, S_matrix=S,
                seed=seed2
            )
    
            reshaped_states = tf.reshape(super_sim_states, [batch_size, M_TRAIN, T_segment + 1, 9])
            reshaped_times  = tf.reshape(super_sim_times,  [batch_size, M_TRAIN, T_segment + 1, 1])
            mean_sim_states = tf.reduce_mean(reshaped_states, axis=1)
            mean_sim_times  = tf.reduce_mean(reshaped_times,  axis=1)
    
            loss_value = compute_loss(
                mean_sim_states, batch_target_states,
                mean_sim_times, batch_target_times,
                lambda_t
            )
    
        grads = tape.gradient(loss_value, trainable_vars)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)  # Preserve the training protocol.
    
        # ---- Metrics computed before the parameter update ----
        bA, bR = tf.exp(log_beta_A), tf.exp(log_beta_R)
        dA, dR = tf.exp(log_delta_A), tf.exp(log_delta_R)
        gC     = tf.exp(log_gamma_C)
    
        ratio_bA_dA = bA / dA
        ratio_bR_dR = bR / dR
        ratio_dA_dR = dA / dR
    
        rpd  = calculate_rpd(log_beta_A, log_beta_R, log_delta_A, log_delta_R, log_gamma_C)
        mape = calculate_mape(tf.stack([bA, bR, dA, dR, gC]), true_params)
    
        # ---- Finite/NaN guard (graph-safe) ----
        grads_finite = tf.reduce_all([tf.reduce_all(tf.math.is_finite(g)) for g in grads])
        ok = tf.math.is_finite(loss_value) & grads_finite
    
        def _apply():
            optimizer.apply_gradients(zip(grads, trainable_vars))
            # Project parameters to the benchmark bounds.
            for log_var, true_log_param in zip(trainable_vars, true_log_params_list):
                log_var.assign(tf.clip_by_value(log_var, true_log_param - log10, true_log_param + log10))
            return tf.constant(True)
    
        did_update = tf.cond(ok, _apply, lambda: tf.constant(False))
    
        return loss_value, rpd, mape, bA, bR, dA, dR, gC, ratio_bA_dA, ratio_bR_dR, ratio_dA_dR, did_update


    for epoch in range(1, num_epochs + 1):
        # LR decay outside (as before)
        if epoch > 0 and epoch % (25 * kk) == 0:
            old_lr = optimizer.learning_rate.numpy()
            lim = 0.1e-2
            new_lr = (old_lr - lim) * learning_rate_decay_factor + lim
            optimizer.learning_rate.assign(new_lr)
    
        # Sample the training batch.
        all_indices = tf.range(start=0, limit=NUM_START_POINTS, dtype=tf.int32)
        batch_indices = tf.random.shuffle(all_indices)[:batch_size]
    
        batch_initial_states = tf.gather(map_initial_states, batch_indices)
        batch_target_states  = tf.gather(map_target_trajectories, batch_indices)
        batch_target_times   = tf.gather(map_target_times, batch_indices)
    
        (loss_value, rpd, mape, bA, bR, dA, dR, gC,
         ratio_bA_dA, ratio_bR_dR, ratio_dA_dR, did_update) = train_step(
            batch_initial_states, batch_target_states, batch_target_times
        )

       
        history['epoch'].append(epoch)
        history['loss'].append(loss_value.numpy())
        history['rpd'].append(rpd.numpy())
        history['mape'].append(mape.numpy())
        history['beta_A'].append(bA.numpy())
        history['beta_R'].append(bR.numpy())
        history['delta_A'].append(dA.numpy())
        history['delta_R'].append(dR.numpy())
        history['gamma_C'].append(gC.numpy())
        history['ratio_bA_dA'].append(ratio_bA_dA.numpy())
        history['ratio_bR_dR'].append(ratio_bR_dR.numpy())
        history['ratio_dA_dR'].append(ratio_dA_dR.numpy())

        current_lr = optimizer.learning_rate.numpy()

        # Calculate Median MAPE
        median_mape_str = f"{'N/A':>7}"
        if len(history['loss']) >= 1:
            num_to_consider = min(5*(num_epochs//20)+1, len(history['loss']))
            idx = np.arange(len(history['loss']))[-num_to_consider:]

            param_keys = ['beta_A', 'beta_R', 'delta_A', 'delta_R', 'gamma_C']
            median_params_list = [np.exp(np.mean(np.log(np.array(history[var])[idx]))) for var in param_keys]
            median_params_tf = tf.constant(median_params_list, dtype=tf.float32)
            median_mape = calculate_mape(median_params_tf, true_params)
            median_mape_str = f"{median_mape.numpy():6.2f}%"
            history['medmape'].append(median_mape.numpy())
        else:
            history['medmape'].append(mape.numpy())

        if epoch % 25 == 0 or epoch == 1:
            print(f"Epoch {epoch:04d} | Loss: {loss_value.numpy():8.3e} | MAPE: {mape.numpy():6.2f}% | MedMAPE: {median_mape_str} | LR: {current_lr:8.2e}")

        if verbose and (epoch % 25 == 0 or epoch == 1):
            print(f"  Ratios  | bA/dA: {ratio_bA_dA.numpy():.2f} (true {true_ratios['bA_dA']:.2f}) | "
                  f"bR/dR: {ratio_bR_dR.numpy():.2f} (true {true_ratios['bR_dR']:.2f}) | "
                  f"dA/dR: {ratio_dA_dR.numpy():.2f} (true {true_ratios['dA_dR']:.2f})")
            print(f"  Params  | beta_A: {bA.numpy():.4f} (true: {bA_true:.4f}) | "
                  f"beta_R: {bR.numpy():.4f} (true: {bR_true:.4f}) | "
                  f"gamma_C: {gC.numpy():.4f} (true: {gC_true:.4f})")


        # Clip parameters to reasonable bounds
        log10 = tf.math.log(10.0)
        for log_var, true_log_param in zip(trainable_vars, true_log_params):
            log_var.assign(tf.clip_by_value(log_var, true_log_param - log10, true_log_param + log10))

    training_time = time.time() - start_time
    print(f"\nTraining finished in {training_time:.2f} seconds.")

    # Compute per-run statistics
    run_stats = compute_run_statistics(history, stats_pct)
    print(f"\n--- Per-run statistics (last {stats_pct}% = {run_stats['num_selected']} epochs) ---")
    print(f"Median params: beta_A={run_stats['median_params']['beta_A']:.4f}, "
          f"beta_R={run_stats['median_params']['beta_R']:.4f}, "
          f"delta_A={run_stats['median_params']['delta_A']:.4f}, "
          f"delta_R={run_stats['median_params']['delta_R']:.4f}, "
          f"gamma_C={run_stats['median_params']['gamma_C']:.4f}")
    print(f"Median MAPE: {run_stats['median_mape']:.4f}%")
    print(f"MedMAPE (MAPE of median params): {run_stats['medmape']:.4f}%")

    print(f"\n=== Run complete (seed={seed}) ===")

    final_params = {
        'beta_A': tf.exp(log_beta_A).numpy(),
        'beta_R': tf.exp(log_beta_R).numpy(), 
        'delta_A': tf.exp(log_delta_A).numpy(),
        'delta_R': tf.exp(log_delta_R).numpy(),
        'gamma_C': tf.exp(log_gamma_C).numpy()
    }

    return history, final_params, run_stats


# ========================================================================
# MULTI-RUN STATISTICS COLLECTION
# ========================================================================

def run_multiple_experiments(N, stats_pct=25.0, random_init=True, base_seed=43, **kwargs):
    """
    Run InferenceOscillator N times and collect statistics.

    Parameters:
    -----------
    N : int
        Number of runs
    stats_pct : float
        Percentage of final epochs to use for statistics (default: 25%)
    random_init : bool
        If True, use random initial multipliers for each run (default: True)
    base_seed : int
        Base seed for generating run-specific seeds (default: 43)
    **kwargs : dict
        Additional parameters passed to InferenceOscillator

    Returns:
    --------
    results : dict
        Contains all histories, per-run stats, global stats, and aggregated stats
    """
    print(f"\n{'='*70}")
    print(f"RUNNING {N} EXPERIMENTS")
    print(f"{'='*70}")

    np.random.seed(base_seed)

    all_histories = []
    all_run_stats = []
    all_init_multipliers = []
    all_final_params = []

    rng = np.random.default_rng(base_seed)
    
    for run_idx in range(N):
        print(f"\n{'='*70}")
        print(f"RUN {run_idx + 1}/{N}")
        print(f"{'='*70}")

        # Generate random initial multipliers if requested
        if random_init:
            #init_multipliers = np.exp(np.random.uniform(-np.log(4), np.log(4), 5))
            init_multipliers = np.exp(rng.uniform(-np.log(4), np.log(4), 5))
        else:
            init_multipliers = np.array([0.25, 4.0, 4.0, 0.25, 0.25])

        # Generate seed for this run
        run_seed = base_seed + run_idx * 97377

        # Run inference
        history, final_params, run_stats = InferenceOscillator(
            init_multipliers=init_multipliers,
            seed=run_seed,
            stats_pct=stats_pct,
            **kwargs
        )

        all_histories.append(history)
        all_run_stats.append(run_stats)
        all_init_multipliers.append(init_multipliers)
        all_final_params.append(final_params)

        gc.collect()

    # -----------------------------------------------------------------------------------
    # Compute global statistics (pooled across all runs)
    # -----------------------------------------------------------------------------------
    print(f"\n{'='*70}")
    print("COMPUTING GLOBAL STATISTICS")
    print(f"{'='*70}")

    param_keys = ['beta_A', 'beta_R', 'delta_A', 'delta_R', 'gamma_C']
    true_vals = np.array([50.0, 5.0, 1.0, 0.2, 2.0])

    # Pool all epochs from all runs
    pooled_data = {key: [] for key in param_keys + ['mape', 'loss']}

    for history in all_histories:
        num_epochs = len(history['loss'])
        num_to_select = max(1, int(num_epochs * stats_pct / 100.0))
        idx = np.arange(num_epochs)[-num_to_select:]

        for key in param_keys + ['mape', 'loss']:
            pooled_data[key].extend(np.array(history[key])[idx].tolist())

    # Compute global statistics
    global_stats = {'median_params': {}}
    global_median_params_list = []

    for var in param_keys:
        sample = np.array(pooled_data[var])
        log_sample = np.log(sample + 1e-20)
        geo_median = np.exp(np.mean(log_sample))
        ci_low, ci_high = np.exp(np.percentile(log_sample, [2.5, 97.5]))
        global_stats['median_params'][var] = {
            'median': geo_median,
            'ci_low': ci_low,
            'ci_high': ci_high
        }
        global_median_params_list.append(geo_median)

    # Global median MAPE
    mape_sample = np.array(pooled_data['mape'])
    global_stats['median_mape'] = {
        'median': np.exp(np.mean(np.log(mape_sample + 1e-20))),
        'ci_low': np.exp(np.percentile(np.log(mape_sample + 1e-20), 2.5)),
        'ci_high': np.exp(np.percentile(np.log(mape_sample + 1e-20), 97.5))
    }

    # Global medmape
    esvals = np.array(global_median_params_list)
    global_stats['medmape'] = 100 * np.mean(np.abs(true_vals - esvals) / true_vals)
    global_stats['total_samples'] = len(pooled_data['mape'])

    # -----------------------------------------------------------------------------------
    # Compute aggregated statistics (mean ± std of per-run statistics)
    # -----------------------------------------------------------------------------------
    aggregated_stats = {'median_params': {}}

    # Aggregate median params
    for var in param_keys:
        per_run_medians = [rs['median_params'][var] for rs in all_run_stats]
        aggregated_stats['median_params'][var] = {
            'mean': np.mean(per_run_medians),
            'std': np.std(per_run_medians)
        }

    # Aggregate median MAPE
    per_run_median_mapes = [rs['median_mape'] for rs in all_run_stats]
    aggregated_stats['median_mape'] = {
        'mean': np.mean(per_run_median_mapes),
        'std': np.std(per_run_median_mapes)
    }

    # Aggregate medmape
    per_run_medmapes = [rs['medmape'] for rs in all_run_stats]
    aggregated_stats['medmape'] = {
        'mean': np.mean(per_run_medmapes),
        'std': np.std(per_run_medmapes)
    }

    # -----------------------------------------------------------------------------------
    # Print summary
    # -----------------------------------------------------------------------------------
    print(f"\n{'='*70}")
    print(f"SUMMARY OF {N} RUNS (using last {stats_pct}% of epochs)")
    print(f"{'='*70}")

    print(f"\n--- Per-Run Statistics (mean ± std) ---")
    for var in param_keys:
        true_val = true_vals[param_keys.index(var)]
        mean_val = aggregated_stats['median_params'][var]['mean']
        std_val = aggregated_stats['median_params'][var]['std']
        print(f"{var:<9s} | {mean_val:8.4f} ± {std_val:8.4f} (true: {true_val:.4f})")

    print(f"{'median_mape':<11s} | {aggregated_stats['median_mape']['mean']:8.4f}% ± {aggregated_stats['median_mape']['std']:8.4f}%")
    print(f"{'medmape':<11s} | {aggregated_stats['medmape']['mean']:8.4f}% ± {aggregated_stats['medmape']['std']:8.4f}%")

    print(f"\n--- Global Statistics (pooled {global_stats['total_samples']} samples) ---")
    for var in param_keys:
        true_val = true_vals[param_keys.index(var)]
        stats = global_stats['median_params'][var]
        print(f"{var:<9s} | Median: {stats['median']:8.4f} | 95% CI: [{stats['ci_low']:8.4f} - {stats['ci_high']:8.4f}] (true: {true_val:.4f})")

    stats = global_stats['median_mape']
    print(f"{'median_mape':<11s} | Median: {stats['median']:8.4f}% | 95% CI: [{stats['ci_low']:8.4f}% - {stats['ci_high']:8.4f}%]")
    print(f"{'medmape':<11s} | {global_stats['medmape']:8.4f}% (MAPE of global median params)")

    results = {
        'histories': all_histories,
        'run_stats': all_run_stats,
        'init_multipliers': all_init_multipliers,
        'final_params': all_final_params,
        'global_stats': global_stats,
        'aggregated_stats': aggregated_stats,
        'N': N,
        'stats_pct': stats_pct
    }

    return results


# ========================================================================
# SAVE/LOAD RESULTS FUNCTIONS
# ========================================================================

def save_results(results, filename='results.pkl'):
    """Save results dictionary to a pickle file."""
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Results saved to {Path(filename).name}")

def load_results(filename='results.pkl'):
    """Load results dictionary from a pickle file."""
    with open(filename, 'rb') as f:
        results = pickle.load(f)
    print(f"Results loaded from {Path(filename).name}")
    return results


# ========================================================================
# PLOTTING FUNCTIONS (All plots consolidated here)
# ========================================================================

def plot_single_run_trajectories(history, final_params, seed=None, T_plot=300000):
    """
    Plot learned vs true trajectories for a single run.
    
    Parameters:
    -----------
    history : dict
        Training history from InferenceOscillator
    final_params : dict
        Final learned parameters
    seed : int, optional
        Seed used for the run (for title)
    T_plot : int
        Number of simulation steps for plotting
    """
    initial_state = np.array([1, 0, 1, 0, 0, 0, 0, 0, 0], dtype=np.float32)
    initial_state_tf = tf.constant(initial_state, dtype=tf.float32)

    print(f"Simulating trajectories for plotting ({T_plot} steps)...")

    np.random.seed(seed)
    tf.random.set_seed(seed)

    seed2 = tf.constant([seed, 11], dtype=tf.int32)

    # Simulate true trajectory
    true_states, true_times = simulate_parallel_fast(
        batch_initial_states=initial_state_tf[None, :], num_steps=T_plot,
        k=tf.constant(k_true), S_matrix=S, seed=seed2
    )
    true_states, true_times = true_states[0], true_times[0]

    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Simulate learned trajectory
    k_learn_final = assemble_k_from_params(
        final_params['beta_A'], final_params['beta_R'],
        final_params['delta_A'], final_params['delta_R'],
        final_params['gamma_C']
    )
    learn_states, learn_times = simulate_parallel_fast(
        batch_initial_states=initial_state_tf[None, :], num_steps=T_plot,
        k=k_learn_final, S_matrix=S, seed=seed2
    )
    learn_states, learn_times = learn_states[0], learn_times[0]

    # Plot trajectories
    title_suffix = ""
    plt.figure(figsize=(8, 8))
    plt.suptitle(f"Learned vs. True Trajectories{title_suffix}", fontsize=14, y=0.95)
    species_map = { 'name': ['Activator (A)', 'Repressor (R)', 'Complex (C)'], 'index': [6, 7, 8] }
    for i in range(3):
        plt.subplot(3, 1, i + 1)
        plt.plot(true_times, true_states[:, species_map['index'][i]], 'b-', label=f"True", alpha=0.8)
        plt.plot(learn_times, learn_states[:, species_map['index'][i]], 'r--', label=f"Learned", alpha=0.8)
        plt.ylabel(f"{species_map['name'][i][0]} count")
        plt.title(species_map['name'][i])
        plt.legend()
        plt.grid(True, alpha=0.4)
        plt.ylim(bottom=0)
    plt.xlabel("Time")
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    for ext in ['.pdf', '.svg', '.png']:
        plt.savefig(f'single_run_trajectories{ext}', transparent=True, bbox_inches='tight')
        
    plt.show()


def plot_single_run_convergence(history, seed=None):
    """
    Plot parameter convergence for a single run.
    
    Parameters:
    -----------
    history : dict
        Training history from InferenceOscillator
    seed : int, optional
        Seed used for the run (for title)
    """
    bA_true, bR_true, dA_true, dR_true, gC_true = k_true[8], k_true[9], k_true[12], k_true[13], k_true[14]
    
    title_suffix = f" (seed={seed})" if seed is not None else ""
    epochs = history['epoch']
    
    plt.figure(figsize=(8, 10))
    plt.suptitle(f"Parameter Convergence{title_suffix}", fontsize=16)
    param_keys = ['beta_A', 'beta_R', 'delta_A', 'delta_R', 'gamma_C']
    param_true_vals = [bA_true, bR_true, dA_true, dR_true, gC_true]
    for i, (key, true_val) in enumerate(zip(param_keys, param_true_vals)):
        plt.subplot(3, 2, i + 1)
        plt.plot(epochs, history[key], 'b-', label='Learned')
        plt.axhline(y=true_val, color='r', linestyle='--', label='Ground Truth')
        plt.title(key)
        plt.xlabel("Epoch")
        plt.ylabel("Value")
        plt.legend()
        plt.grid(True, alpha=0.4)
        plt.ylim(bottom=0)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


def plot_single_run_mape(history, seed=None):
    """
    Plot MAPE convergence for a single run.
    
    Parameters:
    -----------
    history : dict
        Training history from InferenceOscillator
    seed : int, optional
        Seed used for the run (for title)
    """
    title_suffix = f" (seed={seed})" if seed is not None else ""
    
    plt.figure(figsize=(6, 4))
    plt.plot(history['epoch'], history['mape'], 'm-', label='MAPE')
    plt.plot(history['epoch'], history['medmape'], 'b-', label='Median MAPE')
    plt.title(f"MAPE Convergence{title_suffix}")
    plt.xlabel("Epoch")
    plt.ylabel("MAPE (%)")
    plt.grid(True, alpha=0.4)
    plt.yscale('log')
    plt.axhline(y=1, color='r', linestyle='--', label='1% reference')
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_multi_run_parameter_convergence(results):
    """
    Plot parameter convergence across multiple runs.
    
    Parameters:
    -----------
    results : dict
        Output from run_multiple_experiments()
    """
    all_histories = results['histories']
    N = results['N']
    param_keys = ['beta_A', 'beta_R', 'delta_A', 'delta_R', 'gamma_C']
    param_labels = ['$\\' + pk + '$' for pk in param_keys]

    param_true_vals = [50.0, 5.0, 1.0, 0.2, 2.0]

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    fig.suptitle(f'Parameter Convergence Across {N} Runs', fontsize=14)

    for i, (key, true_val, label) in enumerate(zip(param_keys, param_true_vals, param_labels)):
        ax = axes[i // 3, i % 3]
        for run_idx, history in enumerate(all_histories):
            ax.plot(history['epoch'], history[key], alpha=0.5, label=f'Run {run_idx+1}' if i == 0 else None)
        ax.axhline(y=true_val, color='r', linestyle='--', linewidth=2, label='True' if i == 0 else None)
        ax.set_title(label)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)

    # Use last subplot for legend
    axes[1, 2].axis('off')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    axes[1, 2].legend(handles, labels, loc='center', fontsize=10)

    plt.tight_layout()
    
    for ext in ['.pdf', '.svg', '.png']:
        plt.savefig(f'multi_run_parameter_convergence{ext}', transparent=True, bbox_inches='tight')
    
    plt.show()


def plot_multi_run_mape(results):
    """
    Plot MAPE convergence across multiple runs.
    
    Parameters:
    -----------
    results : dict
        Output from run_multiple_experiments()
    """
    all_histories = results['histories']
    N = results['N']

    plt.figure(figsize=(10, 5))
    for run_idx, history in enumerate(all_histories):
        plt.semilogy(history['epoch'], history['mape'], alpha=0.6, label=f'Run {run_idx+1}')
    plt.axhline(y=1, color='r', linestyle='--', linewidth=2, label='1% reference')
    plt.xlabel('Epoch')
    plt.ylabel('MAPE (%)')
    plt.title(f'MAPE Convergence Across {N} Runs')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    for ext in ['.pdf', '.svg', '.png']:
        plt.savefig(f'multi_run_mape{ext}', transparent=True, bbox_inches='tight')
    
    plt.show()


def plot_multi_run_boxplots(results):
    """
    Plot box plots of per-run statistics.
    
    Parameters:
    -----------
    results : dict
        Output from run_multiple_experiments()
    """
    all_run_stats = results['run_stats']
    param_keys = ['beta_A', 'beta_R', 'delta_A', 'delta_R', 'gamma_C']
    param_true_vals = [50.0, 5.0, 1.0, 0.2, 2.0]

    per_run_median_mapes = [rs['median_mape'] for rs in all_run_stats]
    per_run_medmapes = [rs['medmape'] for rs in all_run_stats]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Box plot of median MAPE
    ax = axes[0]
    ax.boxplot([per_run_median_mapes], tick_labels=['Median MAPE'])
    ax.set_ylabel('MAPE (%)')
    ax.set_title('Per-Run Median MAPE')
    ax.grid(True, alpha=0.3)

    # Box plot of medmape
    ax = axes[1]
    ax.boxplot([per_run_medmapes], tick_labels=['MedMAPE'])
    ax.set_ylabel('MAPE (%)')
    ax.set_title('Per-Run MedMAPE')
    ax.grid(True, alpha=0.3)

    # Box plot of all parameters (normalized to true values)
    ax = axes[2]
    normalized_params = []
    for var, true_val in zip(param_keys, param_true_vals):
        per_run_vals = [rs['median_params'][var] / true_val for rs in all_run_stats]
        normalized_params.append(per_run_vals)
    ax.boxplot(normalized_params, tick_labels=param_keys)
    ax.axhline(y=1.0, color='r', linestyle='--', linewidth=1)
    ax.set_ylabel('Normalized Value (learned/true)')
    ax.set_title('Per-Run Median Params (normalized)')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()


def plot_all_multi_run(results):
    """
    Generate all plots for multi-run experiments.
    
    Parameters:
    -----------
    results : dict
        Output from run_multiple_experiments()
    """
    plot_multi_run_parameter_convergence(results)
    plot_multi_run_mape(results)
    plot_multi_run_boxplots(results)


# ========================================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# ========================================================================

import scipy.stats

def bootstrap_medmape_ci(results, n_bootstrap=10000, ci_level=95, block_bootstrap=True):
    """
    Compute bootstrap confidence intervals for medmape.

    Parameters:
    -----------
    results : dict
        Output from run_multiple_experiments()
    n_bootstrap : int
        Number of bootstrap iterations (default: 10000)
    ci_level : float
        Confidence level in percent (default: 95)
    block_bootstrap : bool
        If True, resample entire runs (accounts for within-run correlation).
        If False, resample individual epochs (assumes independence).

    Returns:
    --------
    dict : Contains medmape, ci_low, ci_high, and bootstrap_samples
    """
    param_keys = ['beta_A', 'beta_R', 'delta_A', 'delta_R', 'gamma_C']
    true_vals = np.array([50.0, 5.0, 1.0, 0.2, 2.0])
    stats_pct = results['stats_pct']
    N = results['N']

    # Extract selected epochs per run (keep runs separate for block bootstrap)
    runs_data = []  # List of dicts, one per run

    for history in results['histories']:
        num_epochs = len(history['loss'])
        num_to_select = max(1, int(num_epochs * stats_pct / 100.0))
        idx = np.arange(num_epochs)[-num_to_select:]

        run_params = {}
        for key in param_keys:
            run_params[key] = np.array(history[key])[idx]
        runs_data.append(run_params)

    # Bootstrap
    np.random.seed(43)  # For reproducibility
    bootstrap_medmapes = []

    for _ in range(n_bootstrap):
        if block_bootstrap:
            # Resample entire runs (with replacement)
            boot_run_indices = np.random.choice(N, size=N, replace=True)

            # Pool epochs from selected runs
            pooled_params = {key: [] for key in param_keys}
            for run_idx in boot_run_indices:
                for key in param_keys:
                    pooled_params[key].extend(runs_data[run_idx][key].tolist())

            # Convert to arrays
            for key in param_keys:
                pooled_params[key] = np.array(pooled_params[key])
        else:
            # Pool all data first, then resample individual epochs
            pooled_params = {key: [] for key in param_keys}
            for run_data in runs_data:
                for key in param_keys:
                    pooled_params[key].extend(run_data[key].tolist())

            for key in param_keys:
                pooled_params[key] = np.array(pooled_params[key])

            n_samples = len(pooled_params[param_keys[0]])
            boot_idx = np.random.choice(n_samples, size=n_samples, replace=True)

            for key in param_keys:
                pooled_params[key] = pooled_params[key][boot_idx]

        # Compute geometric median for each parameter
        median_params = []
        for key in param_keys:
            log_sample = np.log(pooled_params[key] + 1e-20)
            geo_median = np.exp(np.mean(log_sample))
            median_params.append(geo_median)

        # Compute medmape for this bootstrap sample
        esvals = np.array(median_params)
        medmape = 100 * np.mean(np.abs(true_vals - esvals) / true_vals)
        bootstrap_medmapes.append(medmape)

    bootstrap_medmapes = np.array(bootstrap_medmapes)

    # Percentile confidence interval
    alpha = (100 - ci_level) / 2
    ci_low, ci_high = np.percentile(bootstrap_medmapes, [alpha, 100 - alpha])

    # Also compute BCa (bias-corrected and accelerated) CI for better coverage
    # This is optional but more accurate for skewed distributions
    original_medmape = results['global_stats']['medmape']

    # Bias correction
    z0 = scipy.stats.norm.ppf(np.mean(bootstrap_medmapes < original_medmape))

    return {
        'medmape': original_medmape,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'std': np.std(bootstrap_medmapes),
        'bootstrap_samples': bootstrap_medmapes,
        'method': 'block' if block_bootstrap else 'standard'
    }


def plot_bootstrap_distribution(ci):
    """
    Plot the bootstrap distribution of MedMAPE.
    
    Parameters:
    -----------
    ci : dict
        Output from bootstrap_medmape_ci()
    """
    plt.figure(figsize=(8, 4))
    plt.hist(ci['bootstrap_samples'], bins=50, density=True, alpha=0.7)
    plt.axvline(ci['medmape'], color='r', linestyle='-', label=f"MedMAPE = {ci['medmape']:.4f}%")
    plt.axvline(ci['ci_low'], color='g', linestyle='--', label=f"95% CI: [{ci['ci_low']:.4f}%, {ci['ci_high']:.4f}%]")
    plt.axvline(ci['ci_high'], color='g', linestyle='--')
    plt.xlabel('MedMAPE (%)')
    plt.ylabel('Density')
    method_name = 'Block Bootstrap' if ci['method'] == 'block' else 'Standard Bootstrap'
    plt.title(f'Bootstrap Distribution of MedMAPE ({method_name})')
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# CHECKPOINTED MULTI-RUN EXECUTION
# ============================================================
PARAM_KEYS_FINAL = ["beta_A", "beta_R", "delta_A", "delta_R", "gamma_C"]
TRUE_VALUES_FINAL = np.array([50.0, 5.0, 1.0, 0.2, 2.0], dtype=float)


def aggregate_completed_runs(run_records, stats_pct):
    """Construct the same result schema as run_multiple_experiments()."""
    histories = [record["history"] for record in run_records]
    run_stats = [record["run_stats"] for record in run_records]
    init_multipliers = [record["init_multipliers"] for record in run_records]
    final_params = [record["final_params"] for record in run_records]

    pooled_data = {
        key: [] for key in PARAM_KEYS_FINAL + ["mape", "loss"]
    }
    for history in histories:
        num_epochs = len(history["loss"])
        num_selected = max(1, int(num_epochs * stats_pct / 100.0))
        selected = np.arange(num_epochs)[-num_selected:]
        for key in PARAM_KEYS_FINAL + ["mape", "loss"]:
            pooled_data[key].extend(
                np.asarray(history[key])[selected].tolist()
            )

    global_stats = {"median_params": {}}
    global_parameters = []
    for key in PARAM_KEYS_FINAL:
        sample = np.asarray(pooled_data[key], dtype=float)
        log_sample = np.log(sample + 1e-20)
        geometric_mean = float(np.exp(np.mean(log_sample)))
        ci_low, ci_high = np.exp(
            np.percentile(log_sample, [2.5, 97.5])
        )
        global_stats["median_params"][key] = {
            "median": geometric_mean,
            "ci_low": float(ci_low),
            "ci_high": float(ci_high),
        }
        global_parameters.append(geometric_mean)

    mape_sample = np.asarray(pooled_data["mape"], dtype=float)
    log_mape = np.log(mape_sample + 1e-20)
    global_stats["median_mape"] = {
        "median": float(np.exp(np.mean(log_mape))),
        "ci_low": float(np.exp(np.percentile(log_mape, 2.5))),
        "ci_high": float(np.exp(np.percentile(log_mape, 97.5))),
    }
    global_stats["medmape"] = float(
        100.0
        * np.mean(
            np.abs(
                np.asarray(global_parameters) - TRUE_VALUES_FINAL
            )
            / TRUE_VALUES_FINAL
        )
    )
    global_stats["total_samples"] = len(mape_sample)

    aggregated_stats = {"median_params": {}}
    for key in PARAM_KEYS_FINAL:
        values = np.asarray(
            [stats["median_params"][key] for stats in run_stats],
            dtype=float,
        )
        aggregated_stats["median_params"][key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
        }

    median_mapes = np.asarray(
        [stats["median_mape"] for stats in run_stats],
        dtype=float,
    )
    medmapes = np.asarray(
        [stats["medmape"] for stats in run_stats],
        dtype=float,
    )
    aggregated_stats["median_mape"] = {
        "mean": float(np.mean(median_mapes)),
        "std": float(np.std(median_mapes)),
    }
    aggregated_stats["medmape"] = {
        "mean": float(np.mean(medmapes)),
        "std": float(np.std(medmapes)),
    }

    return {
        "histories": histories,
        "run_stats": run_stats,
        "init_multipliers": init_multipliers,
        "final_params": final_params,
        "global_stats": global_stats,
        "aggregated_stats": aggregated_stats,
        "N": len(run_records),
        "stats_pct": float(stats_pct),
        "estimator": "PST",
        "forward_sampler": "tf.random.stateless_categorical",
        "backward_surrogate": "softmax(logits)",
        "epochs_per_run": NUM_EPOCHS,
        "base_seed": BASE_SEED,
    }


def expected_run_configuration():
    rng = np.random.default_rng(BASE_SEED)
    configurations = []
    for run_index in range(N_RUNS):
        configurations.append(
            {
                "run_index": run_index,
                "seed": BASE_SEED + run_index * 97377,
                "init_multipliers": np.exp(
                    rng.uniform(-np.log(4), np.log(4), 5)
                ).astype(np.float32),
            }
        )
    return configurations


def validate_checkpoint(record, expected):
    if int(record["run_index"]) != int(expected["run_index"]):
        raise ValueError("Checkpoint run index does not match.")
    if int(record["seed"]) != int(expected["seed"]):
        raise ValueError("Checkpoint seed does not match.")
    if not np.allclose(
        record["init_multipliers"],
        expected["init_multipliers"],
        rtol=0,
        atol=1e-7,
    ):
        raise ValueError("Checkpoint initialization does not match.")
    if len(record["history"]["epoch"]) != NUM_EPOCHS:
        raise ValueError(
            "Checkpoint does not contain the requested 3000 epochs."
        )


if __name__ == "__main__":
    final_path = output_path(OUTPUT_FILE)
    checkpoint_directory = output_path(CHECKPOINT_DIR)
    checkpoint_directory.mkdir(parents=True, exist_ok=True)

    if final_path.is_file() and not OVERWRITE_FINAL:
        print(f"Loading existing final PST result from {final_path.name}...")
        with open(final_path, "rb") as handle:
            existing = pickle.load(handle)
        validate_final_results(existing)
        print("The 10-run, 3000-epoch stateless-categorical PST result is already complete.")
    else:

        configurations = expected_run_configuration()
        completed = []

        for config in configurations:
            run_number = config["run_index"] + 1
            checkpoint_path = (
                checkpoint_directory / f"pst_run_{run_number:02d}.pkl"
            )

            if checkpoint_path.is_file():
                print(f"Loading checkpoint {checkpoint_path.name}...")
                with open(checkpoint_path, "rb") as handle:
                    record = pickle.load(handle)
                validate_checkpoint(record, config)
            else:
                print("\n" + "=" * 72)
                print(f"PST STATELESS-CATEGORICAL 3000-EPOCH RUN {run_number}/{N_RUNS}")
                print("=" * 72)

                history, final_params, run_stats = InferenceOscillator(
                    init_multipliers=config["init_multipliers"],
                    seed=config["seed"],
                    stats_pct=STATS_PCT,
                    kk=1,
                    num_epochs=NUM_EPOCHS,
                )

                record = {
                    **config,
                    "history": history,
                    "final_params": final_params,
                    "run_stats": run_stats,
                }
                atomic_pickle_dump(record, checkpoint_path)
                print(f"Checkpoint saved to {checkpoint_path.name}")

            completed.append(record)
            gc.collect()

        results = aggregate_completed_runs(completed, STATS_PCT)
        validate_final_results(results)
        atomic_pickle_dump(results, final_path)

        print("\n" + "=" * 72)
        print("PST STATELESS-CATEGORICAL 3000-EPOCH EXPERIMENT COMPLETE")
        print("=" * 72)
        print(f"Saved final result to {final_path.name}")
        print(
            "Global median-vector MAPE: "
            f"{results['global_stats']['medmape']:.4f}%"
        )

=== Running Inference Oscillator (seed=43) ===
Initial multipliers: [1.5254099  0.28226092 0.26427618 2.561254   1.2732998 ]
Parameters: 262144 start points, 25 train sims, 3000 epochs
--- Per-run statistics (last 25.0% = 750 epochs) ---
Median params: beta_A=49.1923, beta_R=5.1099, delta_A=0.9837, delta_R=0.2065, gamma_C=1.9875
Median MAPE: 1.8588%
MedMAPE (MAPE of median params): 1.8604%

=== Running Inference Oscillator (seed=97420) ===
Initial multipliers: [0.46613538 2.0099633  0.5193463  0.8010208  0.87304354]
Parameters: 262144 start points, 25 train sims, 3000 epochs
--- Per-run statistics (last 25.0% = 750 epochs) ---
Median params: beta_A=49.1651, beta_R=5.1072, delta_A=0.9840, delta_R=0.2042, gamma_C=1.9478
Median MAPE: 2.0228%
MedMAPE (MAPE of median params): 2.0257%

=== Running Inference Oscillator (seed=194797) ===
Initial multipliers: [3.5338929  2.9641259  0.5413119  0.5411638  0.80552334]
Parameters: 262144 start points, 25 train sims, 3000 epochs
--- Per-run statisti